# Topic: Collaborative Filtering & Matrix Factorization

## Definition (30-second explanation)
* A recommendation system technique that finds similar users or items based on past interactions (ratings, clicks, purchases).
* It operates on the core insight: "people who agreed in the past will agree in the future".
* It strictly relies on the user-item interaction matrix and does NOT require item features like genre or description.

## Why Interviewers Ask This
* It is the most popular and foundational recommendation technique used in the industry.
* It tests a candidate's ability to handle massive data scaling and sparsity problems (e.g., 10M users with few ratings).

## Core Concepts
* **Memory-Based CF:** Finds similarities directly from raw data; can be User-User (find similar users) or Item-Item (find similar items).
* **Model-Based CF:** Uses machine learning algorithms to decompose the rating matrix into hidden structures, heavily relying on Matrix Factorization.
* **Latent Factors (k):** Hidden properties (like genre, mood, or director style) inferred purely from interaction patterns.
* **Sparsity:** The real-world problem where most users have rated only a tiny fraction of the available catalog.

## When to Use
* When you possess sufficient historical user-item interaction data.
* When you need a domain-agnostic system that works without requiring rich metadata or descriptions of the items.

## Advantages
* Highly versatile because it works across any domain (movies, products, articles) without needing item features.
* Model-based methods efficiently handle data sparsity and scale well to massive datasets.

## Limitations
* **Cold Start Problem:** It fails for new users or items because they lack the historical interactions needed to find similarities.
* Memory-based approaches (especially User-User) are highly memory intensive and do not scale easily.

## Common Comparisons
* **User-User vs. Item-Item:** Item-Item generally scales better because product catalogs are typically smaller and less dynamic than user bases.
* **Memory-Based vs. Model-Based:** Memory-based is simple and interpretable, whereas Model-based handles sparsity better and is highly scalable.

## Common Interview Traps
* Forgetting to address the cold start problem, which is the biggest weakness of CF.
* Computing cosine similarity on raw ratings without first centering the data by subtracting the user's mean rating.
* Failing to normalize the ratings matrix prior to applying factorization.
* Treating implicit feedback (like clicks or watch time) with the same absolute confidence as explicit feedback (like star ratings).

## Python / SQL Syntax
```python
from surprise import SVD, Dataset, Reader
from surprise.model_selection import cross_validate

# 1. Define rating scale and load dataframe
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df[['userID', 'itemID', 'rating']], reader)

# 2. Build and train SVD model (Matrix Factorization)
svd = SVD(n_factors=50, n_epochs=20, lr_all=0.005, reg_all=0.02)

# 3. Cross-validate to evaluate (RMSE, MAE)
results = cross_validate(svd, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)

# 4. Train on full dataset and predict
trainset = data.build_full_trainset()
svd.fit(trainset)
pred = svd.predict('U1', 'I4')
```

## Important Formula
**Matrix Factorization Equation:** 
R ≈ P × Q^T

*   **R:** The sparse User-Item ratings matrix (n_users × n_items).
*   **P:** The User latent factor matrix (n_users × k).
*   **Q:** The Item latent factor matrix (n_items × k).

## 45-Second Interview Answer
"Collaborative filtering powers recommendations by relying purely on historical interactions rather than item metadata, essentially assuming that users who agreed in the past will agree in the future. While memory-based approaches are easy to interpret, they suffer at scale. In production, I would rely on model-based Matrix Factorization—often deploying Alternating Least Squares on Spark to decompose the sparse user-item matrix into latent factors. However, because CF inherently suffers from the cold start problem, I would always pair it with a fallback strategy for new users."

## Practice Questions:

### Q1: What is the cold start problem in collaborative filtering?
**Answer:**
The cold start problem occurs when new users or new items enter the system with no prior interaction history (ratings, clicks, purchases). Because collaborative filtering relies entirely on these past interactions to find similarities, it completely fails to generate recommendations for these new entities.

**Common Mistakes Candidates Make:**
* Stating that CF can figure out new items based on their features (which is Content-Based filtering, not CF).
* Not explicitly mentioning that it applies to BOTH new users and new items.

**Likely Interviewer Follow-up:**
How do you modify a Matrix Factorization system to incorporate implicit feedback (like clicks) to help alleviate the cold start for users who browse but don't rate? (Answer: You use a modified algorithm like ALS for Implicit Feedback, which treats implicit actions (clicks/views) not as absolute ratings, but as a "confidence level" that the user prefers the item.)

### Q2: What is the difference between user-user and item-item collaborative filtering?
**Answer:**
User-user CF finds similar people to recommend what they liked, while item-item CF finds similar products based on overlapping user interaction. Item-item generally scales much better because the number of items is usually smaller and more static than the number of active users. 

**Common Mistakes Candidates Make:**
* Failing to articulate the scaling and memory differences between the two.
* Assuming user-user is always better just because personalization is "user-centric."

**Likely Interviewer Follow-up:**
If item-item scales better, in what rare business scenario would user-user CF actually be preferable? (Answer: User-User CF is preferable in highly social or community-driven networks (like dating apps or finding gaming teammates) where the number of users is relatively small and user similarity is the actual product.)

### Q3: How does Matrix Factorization solve the sparsity problem?
**Answer:**
In large systems, most users rate very few items, leaving the interaction matrix extremely sparse. Matrix Factorization solves this by decomposing the massive sparse matrix into two dense, lower-rank matrices representing users and items in a shared latent space. By computing the dot product of these dense vectors, the model mathematically infers and fills in the missing values.

**Common Mistakes Candidates Make:**
* Confusing Matrix Factorization with simple data imputation (like filling missing values with zeroes or averages).
* Forgetting to mention the creation of lower-rank dense matrices.

**Likely Interviewer Follow-up:**
What happens to the model's performance and generalization if you increase the number of latent factors (k) too much? (Answer: Increasing $k$ too much leads to severe overfitting; the model essentially memorizes the sparse training data and completely fails to generalize to new interactions.)

### Q4: What are latent factors and what do they represent?
**Answer:**
Latent factors (k) are hidden, underlying properties derived mathematically from the interaction matrix, projecting users and items into a shared dimensional space. While the model doesn't explicitly label them, in a movie dataset, they organically represent abstract concepts like mood, genre leaning, or a specific director's style.

**Common Mistakes Candidates Make:**
* Confusing latent factors with explicit metadata (e.g., saying "Factor 1 is Action, Factor 2 is Comedy"). They are abstract vectors, not hardcoded categories.

**Likely Interviewer Follow-up:**
How do you determine the optimal number of latent factors for your recommendation system? (Answer: You treat $k$ as a hyperparameter and use grid search with cross-validation, choosing the value that minimizes an offline metric like RMSE on a hold-out test set.)

### Q5: How would you scale collaborative filtering to millions of users?
**Answer:**
At the scale of millions of users, memory-based CF is too computationally heavy. I would scale the system using Model-Based Matrix Factorization, specifically utilizing Alternating Least Squares (ALS). ALS is highly scalable because it alternates fixing the user and item matrices, allowing the optimization process to be heavily parallelized across distributed systems like Apache Spark.

**Common Mistakes Candidates Make:**
* Recommending User-User cosine similarity matrices for 10 million users, which will cause immediate Out-Of-Memory errors.
* Mentioning SVD but failing to mention distributed frameworks like Spark or algorithms like ALS designed for production scaling.

**Likely Interviewer Follow-up:**
How does ALS mathematically differ from traditional Gradient Descent when optimizing the cost function in distributed environments? (Answer: ALS fixes one matrix (e.g., users) to solve for the other (items) linearly, making the math strictly convex and allowing it to be computed in parallel across a distributed cluster (like Spark), whereas standard SGD updates sequentially.)

### Q6: Coding - SQL Item-Item Co-occurrence:
You have an e-commerce database. Write a SQL query to find the top 3 most frequently "co-purchased" items for a specific target item (e.g., item_id = 101). A co-purchase means the same user bought both items.

**Mock Schema**
```sql
CREATE TABLE purchases (
    user_id INT,
    item_id INT
);

INSERT INTO purchases (user_id, item_id) VALUES
(1, 101), (1, 102), (1, 104),
(2, 101), (2, 103),
(3, 101), (3, 102), (3, 103),
(4, 102), (4, 105);
```

**Answer**
```sql
```sql
-- Approach 1: Readable CTEs (Good for communication)
WITH item_table AS (
    SELECT user_id FROM purchases WHERE item_id = 101
),
item_item AS (
    SELECT p.item_id AS co_item
    FROM item_table i 
    JOIN purchases p ON i.user_id = p.user_id
    WHERE p.item_id != 101
)
SELECT co_item, COUNT(co_item) AS co_purchase_count
FROM item_item
GROUP BY co_item
ORDER BY co_purchase_count DESC
LIMIT 3;

-- Approach 2: Concise Self-Join (Fastest to write)
SELECT p2.item_id AS co_item, COUNT(*) AS co_purchase_count
FROM purchases p1
JOIN purchases p2 ON p1.user_id = p2.user_id
WHERE p1.item_id = 101 AND p2.item_id != 101
GROUP BY p2.item_id
ORDER BY co_purchase_count DESC
LIMIT 3;
```

**Interview Tips:**

- Always remember to exclude the anchor item (!= 101), or the top recommendation will just be the item itself!
- Mention verbally: "This SQL logic is the exact foundational math behind Memory-Based Item-Item Collaborative Filtering—calculating co-occurrence frequency."

### Q7: Handling Sparsity with Implicit Feedback
**Scenario:** 95% of users don't leave explicit 1-5 star ratings. How do you adapt Matrix Factorization to use implicit behavioral data (clicks, watch time) to solve sparsity?

**Answer:**
To solve this massive sparsity, I would shift from standard Matrix Factorization to **ALS for Implicit Feedback**. Instead of treating missing values as zeros or trying to map a "click" to a "star rating," this approach introduces two distinct concepts:
1. **Preference (p):** A binary value (1 if the user interacted with the item, 0 if not).
2. **Confidence (c):** A continuous value measuring the strength of the interaction (e.g., 5 minutes of watch time vs. 2 hours of watch time, or an explicit rating which carries the highest confidence).
The algorithm factorizes the matrix by weighting the preference by the confidence score, allowing the model to learn from massive amounts of noisy, implicit browsing behavior without treating it as an absolute guarantee of user satisfaction.

**Interview Tips:**
*   **Keywords:** ALS for Implicit Feedback, Preference vs. Confidence.
*   Never suggest mapping implicit data to explicit data (e.g., "A click equals 3 stars, a full watch equals 5 stars"). That is a massive red flag in interviews. It must be treated as a confidence weight.

### Q8:
Using Python and Pandas, write a short script that takes the raw ratings dataframe below, calculates each user's mean rating, subtracts it to create a "mean-centered" ratings dataframe, and then fills any remaining NaN values with 0.

In [30]:
# Data:
import pandas as pd
import numpy as np

# Mock raw ratings matrix (Users = rows, Items = columns)
# NaN means the user hasn't rated the item
data = {
    'Item_A': [5.0, 3.0, np.nan],
    'Item_B': [4.0, np.nan, 2.0],
    'Item_C': [np.nan, 2.0, 5.0]
}
raw_ratings = pd.DataFrame(data, index=['User_1', 'User_2', 'User_3'])

In [31]:
raw_ratings

,Item_A,Item_B,Item_C
User_1,5.0,4.0,NaN
User_2,3.0,NaN,2.0
User_3,NaN,2.0,5.0


In [32]:
# 1-Line Vectorized Solution:
# Calculate row means (axis=1), subtract them from the dataframe aligning on rows (axis=0), and fill NaNs.
centered_ratings = raw_ratings.sub(raw_ratings.mean(axis=1), axis=0).fillna(0)

In [33]:
centered_ratings

,Item_A,Item_B,Item_C
User_1,0.5,-0.5,0.0
User_2,0.5,0.0,-0.5
User_3,0.0,-1.5,1.5


**Interview Tips:**

- Why we do this: A user who rates everything a 5 and a user who rates everything a 3 might actually like the same movies, but their baseline strictness is different. Mean-centering normalizes for "harsh" vs "generous" raters.

- The Coding Trap: Never use a for loop to iterate over rows in Pandas during an interview. Always use vectorized methods like .sub(), .add(), or .mul() for broadcasting.

### Q9: Why use an Ensemble for Recommendation Systems?
**Scenario:** Why is a single Matrix Factorization model insufficient in production, and how does an ensemble approach (like the Netflix Prize winner) improve recommendations?

**Answer:**
Relying on a single Matrix Factorization (MF) model is risky because every algorithm has blind spots. For instance, MF is incredible at finding deep, global latent patterns, but it struggles with the cold start problem and often misses hyper-local "neighborhood" similarities (e.g., a user exclusively watching one specific anime franchise). 

To build a robust production system, you use an **Ensemble**. You combine multiple models—like ALS Matrix Factorization for global patterns, an Item-Item K-Nearest Neighbors (KNN) model for localized similarities, and a Content-Based Gradient Boosting model (like XGBoost) to handle new items and user metadata. By blending these predictions, the ensemble covers the individual weaknesses of each model, which is exactly how the famous BellKor team won the $1M Netflix Prize.

**Interview Tips:**
*   **The Keyword:** "Ensemble" or "Blending".
*   **The "Why":** Explain that ensembles combine *Global models* (MF) with *Local models* (KNN) and *Feature models* (Content-based) to cover all edge cases and solve the cold start problem.

### Q10: Implementing Collaborative Filtering (Surprise Library)
**Scenario:** You have a Pandas DataFrame `sales_df` containing historical ratings. Write the Python script to train a Matrix Factorization model and predict Customer 99's rating for 'Product X'.

In [46]:
# Data:
import pandas as pd
from surprise import Reader, Dataset, SVD

# ==========================================
# 0. The Mock Data (sales_df)
# ==========================================
# We give Customer 99 and Product X some history so the model can learn
mock_data = {
    'customer_id': [99, 99, 101, 101, 102, 103],
    'product_id': ['Product A', 'Product B', 'Product X', 'Product A', 'Product X', 'Product B'],
    'rating_score': [4.0, 5.0, 3.0, 4.0, 2.0, 5.0]
}
sales_df = pd.DataFrame(mock_data)

In [47]:
# ==========================================
# 1. Data Prep (Surprise Formatting)
# ==========================================
reader = Reader(rating_scale=(1, 5)) 
data = Dataset.load_from_df(sales_df[['customer_id', 'product_id', 'rating_score']], reader)
trainset = data.build_full_trainset()

# ==========================================
# 2. Model Execution
# ==========================================
model = SVD()
model.fit(trainset)

# Predict what Customer 99 would rate Product X
prediction = model.predict(99, 'Product X')

print("Predicted Score for Customer 99 on Product X:", round(prediction.est, 2))

Predicted Score for Customer 99 on Product X: 3.74


**Interview Tips:**

- The "Gotcha": You cannot pass a raw Pandas DataFrame directly into model.fit(). If you do this in a live coding interview, it will throw an error.

- Why we use Reader: The algorithm needs to know the absolute minimum and maximum bounds of the ratings (e.g., 1 to 5) so it doesn't mathematically predict a 7 or a -2.

- Column Order Requirement: The load_from_df function strictly expects exactly three columns in this exact order: User, Item, Rating.

- Verbal pivot: If you forget the exact syntax for Reader or Dataset during an interview, tell the interviewer: "I know the Surprise library requires converting the dataframe into its specific sparse trainset format with a defined rating scale, but I would need to quickly check the docs for the exact instantiation syntax before calling SVD().fit()."